# Step 3: ELT Pipeline Implementation — Olist Data Warehouse Project

**Owner:** Step 3 / ELT Pipeline  
**Tooling:** BigQuery + dbt + SQL  
**Scope:** Transform raw Olist tables into cleaned staging views and star schema warehouse tables.

This notebook documents the Step 3 ELT implementation only.  
It covers dbt source definitions, staging models, warehouse marts, testing, execution, and design decisions.

The customer RFM / repeat-buyer analysis is treated as a downstream analytics step and is not included in this Step 3 marts layer.

# 1. Business Context

## Business problem

Olist wants to understand customer purchasing behaviour and support future analysis of repeat buyers, revenue trends, and customer value. This segment focuses on preparing the warehouse foundation for this analysis.

## Step 3 responsibility

The ELT pipeline is responsible for:

- Reading raw Olist tables from BigQuery
- Cleaning and standardising raw fields in staging models
- Creating a star schema with dimension and fact tables
- Adding dbt tests to validate key fields and relationships
- Producing reliable warehouse tables for later Python/business analysis


## 2. ELT Pipeline Flow

                OLIST RAW DATA
        (customers, orders, products...)

                     │
                     ▼

            BigQuery Raw Dataset
                 (olist_raw)

                     │
                     ▼

              dbt Sources (sources.yml)

                     │
                     ▼

            Staging Layer (Views)
    ┌───────────────────────────────┐
    │ stg_customers                 │
    │ stg_orders                    │
    │ stg_order_items               │
    │ stg_payments                  │
    │ stg_products                  │
    │ stg_sellers                   │
    │ stg_reviews                   │
    └───────────────────────────────┘

                     │
                     ▼

          Star Schema (Tables)
    ┌───────────────────────────────┐
    │ dim_customers                 │
    │ dim_products                  │
    │ dim_sellers                   │
    │ dim_date                      │
    │ fact_orders                   │
    │ • total_sale_amount           │
    └───────────────────────────────┘

                     │
                     ▼

            Data Quality Testing
        not_null • unique • FK • dbt_expectations

                     │
                     ▼

            Python Analytics (Step 5)

# 3. Environment and Setup

## 3.1 Activate the project environment

The project was run from a conda environment named `olist`.

```bash
conda activate elt
```

## Why this matters

A dedicated environment ensures that dbt, the BigQuery adapter, and Python dependencies are isolated from other projects. This reduces version conflicts and makes the project easier to reproduce.


In [ ]:
# Terminal command, shown here for documentation
# conda activate elt

## 3.2 Navigate to the dbt project folder

The ELT work was implemented inside the `dbt_olist` folder.

```bash
cd dbt_olist
```

This folder contains the dbt project files such as:

```text
dbt_project.yml
profiles.yml
sources.yml
models/
  staging/
  marts/
```


In [ ]:
# Terminal command, shown here for documentation
# cd dbt_olist

## 3.3 Validate the dbt project before running transformations

Before running the pipeline, the project was parsed.

```bash
dbt parse
```

## Rationale

`dbt parse` checks the project without creating tables or views in BigQuery.

It validates:

- SQL syntax
- Jinja syntax such as `{{ ref(...) }}` and `{{ source(...) }}`
- model dependencies
- `sources.yml`
- `schema.yml`

This step helps catch configuration or syntax issues early.

## Result

The project parsed successfully, meaning dbt could understand all model files and dependencies.


In [ ]:
# Terminal command, shown here for documentation
# dbt parse

# 4. dbt Source Configuration

## What was done

The raw Olist tables were registered in `sources.yml`.

This allows staging models to reference raw tables using dbt's `source()` function.

## Why this was done

Using `source()` is preferred because it avoids hardcoding raw BigQuery table paths in every SQL model.

Instead of writing a full BigQuery path repeatedly, we can use:

```sql
{{ source('olist_raw', 'customers') }}
```

If the raw dataset changes later, only `sources.yml` needs to be updated.


In [ ]:
# sources.yml
version: 2

sources:
  - name: olist_raw
    database: olist-assignment-497915
    schema: olist_raw
    tables:
      - name: customers
      - name: orders
      - name: order_items
      - name: order_payments
      - name: products
      - name: sellers
      - name: order_reviews

## Source table relevance to the ELT pipeline

| Source table | Why it matters |
|---|---|
| `customers` | Provides customer identifiers and customer location |
| `orders` | Provides order status and purchase timestamps |
| `order_items` | Provides product, seller, price, and freight values at item level |
| `order_payments` | Cleaned in staging for downstream payment/revenue analysis |
| `products` | Provides product attributes for the product dimension |
| `sellers` | Provides seller location for the seller dimension |
| `order_reviews` | Cleaned in staging for optional review analysis |

The geolocation file was not included in the current warehouse scope because the agreed star schema uses customer and seller ZIP/state fields directly.

# 5. Staging Layer

## What was done

The staging layer was created to clean and standardize raw source tables before they flow into final warehouse tables.

Staging models created:

```text
stg_customers
stg_orders
stg_order_items
stg_payments
stg_products
stg_sellers
stg_reviews
```

## Why staging is needed

Raw ingestion preserves source data exactly as delivered.  
Staging makes the data usable by:

- trimming IDs
- standardising string case
- casting numeric fields
- casting timestamps
- filtering clearly invalid values
- preparing fields for downstream joins

## 5.1 `stg_customers`

### Purpose

Clean customer identifiers and customer location fields.

### Final model logic

In [ ]:
{{ config(materialized='view') }}

select
    trim(customer_id) as customer_id,
    trim(customer_unique_id) as customer_unique_id,
    cast(customer_zip_code_prefix as int64) as customer_zip_code_prefix,
    lower(trim(customer_city)) as customer_city,
    upper(trim(customer_state)) as customer_state
from {{ source('olist_raw', 'customers') }}
where customer_id is not null
  and customer_unique_id is not null

### Field rationale

| Field | Why it was kept or created | How it helps solve the problem |
|---|---|---|
| `customer_id` | Order-level customer key used by the orders table | Allows orders to be joined back to customer records |
| `customer_unique_id` | Real customer identifier across multiple orders | Essential for repeat-buyer analysis because repeat purchases must be grouped by the same real customer |
| `customer_zip_code_prefix` | Location attribute | Supports geographic analysis |
| `customer_city` | Standardized to lowercase and trimmed | Avoids duplicate city values caused by inconsistent formatting |
| `customer_state` | Standardized to uppercase and trimmed | Supports reliable state-level analysis |

`customer_unique_id` is the most important customer field because RFM requires one row per real customer.

## 5.2 `stg_orders`

### Purpose

Clean order records and create a date field for time-based customer analysis.

### Final model logic

In [ ]:
{{ config(materialized='view') }}

select
    trim(order_id) as order_id,
    trim(customer_id) as customer_id,
    lower(trim(order_status)) as order_status,
    cast(order_purchase_timestamp as timestamp) as order_purchase_timestamp,
    date(cast(order_purchase_timestamp as timestamp)) as order_purchase_date,
    cast(order_approved_at as timestamp) as order_approved_at,
    cast(order_delivered_carrier_date as timestamp) as order_delivered_carrier_date,
    cast(order_delivered_customer_date as timestamp) as order_delivered_customer_date,
    cast(order_estimated_delivery_date as timestamp) as order_estimated_delivery_date
from {{ source('olist_raw', 'orders') }}
where order_id is not null
  and customer_id is not null

### Field rationale

| Field | Why it was kept or created | How it helps the warehouse |
|---|---|---|
| `order_id` | Unique order identifier | Links orders to payments, items, and reviews |
| `customer_id` | Customer key on the order | Allows orders to be joined to customers |
| `order_status` | Standardized status field | Allows filtering to completed purchases only |
| `order_purchase_timestamp` | Exact order purchase time | Preserves full transaction timing |
| `order_purchase_date` | New date-only field from timestamp | Required for recency calculation in RFM |
| `order_approved_at` | Order approval timestamp | Useful for future operational analysis |
| `order_delivered_carrier_date` | Carrier delivery timestamp | Useful for future delivery analysis |
| `order_delivered_customer_date` | Customer delivery timestamp | Useful for future delivery performance analysis |
| `order_estimated_delivery_date` | Estimated delivery timestamp | Useful for future delay analysis |

`order_purchase_date` was created because recency is calculated as analysis date minus last purchase date.

## 5.3 `stg_payments`

### Purpose

Clean payment records and prepare payment value for revenue calculations.

### Final model logic

In [ ]:
{{ config(materialized='view') }}

select
    trim(order_id) as order_id,
    cast(payment_sequential as int64) as payment_sequential,
    lower(trim(payment_type)) as payment_type,
    cast(payment_installments as int64) as payment_installments,
    cast(payment_value as numeric) as payment_value
from {{ source('olist_raw', 'order_payments') }}
where order_id is not null
  and payment_value is not null
  and cast(payment_value as numeric) >= 0

### Field rationale

| Field | Why it was kept or created | How it helps the warehouse |
|---|---|---|
| `order_id` | Links payment to order | Allows payments to be linked to orders for downstream analysis |
| `payment_sequential` | Preserves payment sequence | Helps identify orders with multiple payment rows |
| `payment_type` | Standardized payment method | Enables optional payment behavior analysis |
| `payment_installments` | Number of installments used | Can help analyze whether installment usage affects spending |
| `payment_value` | Cast to numeric | Required for revenue and monetary value calculations |

Payment data needs aggregation because one order can have multiple payment rows.

## 5.4 `stg_order_items`

### Purpose

Clean order item records and standardise item-level price and freight fields.

### Final model logic

In [ ]:
{{ config(materialized='view') }}

select
    trim(order_id) as order_id,
    cast(order_item_id as int64) as order_item_id,
    trim(product_id) as product_id,
    trim(seller_id) as seller_id,
    cast(shipping_limit_date as timestamp) as shipping_limit_date,
    cast(price as numeric) as price,
    cast(freight_value as numeric) as freight_value
from {{ source('olist_raw', 'order_items') }}
where order_id is not null
  and product_id is not null
  and seller_id is not null

### Field rationale

| Field | Why it was kept or created | How it helps the warehouse |
|---|---|---|
| `order_id` | Connects item to order | Allows item-level data to join to order/customer data |
| `order_item_id` | Item number inside each order | Used to create an order-item grain fact table |
| `product_id` | Product key | Connects order item to product dimension |
| `seller_id` | Seller key | Connects order item to seller dimension |
| `shipping_limit_date` | Shipping deadline | Useful for future operational analysis |
| `price` | Item price | Core sales measure |
| `freight_value` | Shipping cost | Supports freight and item cost analysis |

The derived total field is created later in `fact_orders`, where the transactional fact table is assembled.

## 5.5 `stg_products`, `stg_sellers`, and `stg_reviews`

These models support the wider warehouse and future analyses.

| Model | Purpose | Relevance to repeat-buyer problem |
|---|---|---|
| `stg_products` | Cleans product attributes such as category and dimensions | Allows future analysis of which product categories drive repeat purchase |
| `stg_sellers` | Cleans seller location details | Allows future seller-retention analysis |
| `stg_reviews` | Cleans review scores and comments | Allows future analysis of satisfaction and repeat purchase behavior |

These tables are not the core of RFM, but they make the warehouse more complete and support additional business questions.


# 6. Star Schema Recap (from Phase 2)

## What was done

The team’s agreed star schema was implemented in the marts layer.

Dimension tables:

```text
dim_customers
dim_products
dim_sellers
dim_date
```

Fact table:

```text
fact_orders
```

## Why a star schema was used

A star schema separates measurable business events from descriptive attributes.

This makes the warehouse easier to query because analysts can start from the fact table and join to dimensions for customer, product, seller, and date attributes.

## Important grain note

Although the table is named `fact_orders`, the actual grain is:

```text
1 row = 1 item within an order
```

This is because the fact table is built from `order_items`, where each order can contain one or more products.

## 7.1 `fact_orders`

### Purpose

Create a central transactional fact table at order-item level.

### Grain

```text
1 row = 1 item within an order
```

This means one order with three items will have three rows in `fact_orders`.

### Final model logic


In [ ]:
{{ config(materialized='table') }}

with staging_order_items as (
    select * from {{ ref('stg_order_items') }}
),
staging_orders as (
    select * from {{ ref('stg_orders') }}
)
select
    concat(i.order_id, '-', cast(i.order_item_id as string)) as order_item_sk,
    i.order_id as order_key,
    i.product_id as product_key,
    i.seller_id as seller_key,
    o.customer_id as customer_key,
    cast(format_date('%Y%m%d', date(o.order_purchase_timestamp)) as string) as date_key,
    i.price,
    i.freight_value,
    (i.price + i.freight_value) as total_sale_amount
from staging_order_items i
left join staging_orders o
    on i.order_id = o.order_id
where o.order_id is not null

### Field rationale

| Field | Why it was created | How it helps the warehouse |
|---|---|---|
| `order_item_sk` | Unique surrogate key using `order_id + order_item_id` | Ensures each fact row is uniquely identifiable |
| `order_key` | Foreign key to order | Preserves the original order reference |
| `product_key` | Foreign key to product dimension | Enables product/category analysis |
| `seller_key` | Foreign key to seller dimension | Enables seller performance analysis |
| `customer_key` | Foreign key to customer dimension | Connects purchases to customers |
| `date_key` | Foreign key to date dimension | Enables time-based analysis |
| `price` | Item price | Core item-level sales metric |
| `freight_value` | Shipping cost | Supports logistics and cost analysis |
| `total_sale_amount` | New derived measure: `price + freight_value` | Total sales amount column for downstream analysis |

Important: `fact_orders` supports the warehouse design at order-item grain. Customer-level totals, customer spend over time, and RFM segmentation are downstream analytics outputs and are therefore not included in the Step 3 marts layer.

# 8. Data Quality Testing with `schema.yml`

## What was done

Data tests were added to validate important staging, dimension, and fact table fields.

Test types used:

```text
not_null
unique
relationships
dbt_expectations tests
```

## Why tests were needed

The star schema is only useful if the underlying joins, keys, and measures are reliable.

Examples:

- `order_id` should not be null in staging orders
- dimension primary keys should be unique
- `fact_orders.customer_key` should successfully map to `dim_customers.customer_key`
- `fact_orders.product_key` should successfully map to `dim_products.product_key`
- `fact_orders.seller_key` should successfully map to `dim_sellers.seller_key`
- `fact_orders.date_key` should successfully map to `dim_date.date_key`
- `fact_orders.total_sale_amount` should be positive because it represents item price plus freight


# 9. Pipeline Execution and Validation

## Parse project

```bash
dbt parse
```

Result:

```text
Completed successfully
```

## Build models

```bash
dbt run
```

Final build result:

```text
PASS=12
WARN=0
ERROR=0
SKIP=0
TOTAL=12
```

The run created:

```text
5 table models
7 view models
```

## Run tests

```bash
dbt test
```

Expected final result after:
1. removing the `dim_date.year` set test, and
2. adding the `total_sale_amount` test:

```text
PASS=60
WARN=1
ERROR=0
SKIP=0
TOTAL=61
```

The remaining warning is acceptable because `stg_products.product_weight_g` is configured with `severity: warn`. It indicates that 4 product rows have unexpected product weight values, but it does not block the warehouse build.

In [ ]:
# dbt command used for validation
# dbt parse
# dbt run
# dbt test

# 10. Pipeline Execution and Validation

## Parse project

```bash
dbt parse
```

Result:

```text
Completed successfully
```

## Build models

```bash
dbt run
```

Final build result:

```text
PASS=12
WARN=0
ERROR=0
SKIP=0
TOTAL=12
```

The run created:

```text
5 table models
7 view models
```

## Run tests

```bash
dbt test
```

Expected final result after removing the `dim_date.year` set test:

```text
PASS=59
WARN=1
ERROR=0
SKIP=0
TOTAL=60
```

The remaining warning is acceptable because `stg_products.product_weight_g` is configured with `severity: warn`. It indicates that 4 product rows have unexpected product weight values, but it does not block the warehouse build.

# 11. How the ELT Pipeline Supports the Project

## Project problem statement

```text
Which customers are most likely to become repeat buyers?
```

## Step 3 contribution

Step 3 does not directly perform the final repeat-buyer segmentation.

Instead, it prepares the clean warehouse foundation needed for that analysis:

```text
1. Clean raw customer, order, product, seller, payment, and review data
2. Standardize types and formats
3. Build reusable dimension tables
4. Build an order-item level fact table
5. Validate keys, relationships, and core data quality
6. Provide trusted tables for Python analysis and reporting
```

The downstream analysis phase can then use these warehouse tables to calculate customer behaviour, revenue trends, product performance, and repeat-purchase indicators.

# 12. Business Value

The completed Step 3 ELT pipeline enables Olist to:

- Query clean customer, product, seller, order, and date data
- Analyze sales at order-item level
- Join facts to dimensions reliably
- Build monthly sales trends
- Analyze top-selling products
- Support future customer segmentation and repeat-buyer analysis
- Reduce manual cleaning work for analysts
- Improve trust in downstream reporting